# RAG baseline evaluation — *Attention Is All You Need*

This notebook evaluates the existing RAG implementation against Salma's real processed `DocumentChunk` JSON. It does not alter production RAG code or re-chunk the paper.

Install dependencies first:

```bash
pip install -r backend/requirements.txt
# rank-bm25 is included for the optional hybrid experiment
```

In [2]:
from __future__ import annotations
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
import json, sys, textwrap

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').exists(): raise RuntimeError(f'Could not find repository root from {PROJECT_ROOT}.')
sys.path.insert(0, str(PROJECT_ROOT))
CHUNKS_JSON_PATH = Path(r'C:\Users\Huawei\Downloads\response_1788952144966.json')
if not CHUNKS_JSON_PATH.exists(): raise FileNotFoundError(f'Chunk JSON not found: {CHUNKS_JSON_PATH}')
print(f'Using Salma chunk JSON: {CHUNKS_JSON_PATH}')

Using Salma chunk JSON: C:\Users\Huawei\Downloads\response_1788952144966.json


In [3]:
REQUIRED_FIELDS = {'document_id', 'chunk_id', 'page_number', 'section', 'text'}
raw_chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding='utf-8'))
if not isinstance(raw_chunks, list) or not raw_chunks: raise ValueError('Expected a non-empty list of DocumentChunk JSON objects.')
if any(not isinstance(item, dict) or REQUIRED_FIELDS - item.keys() for item in raw_chunks): raise ValueError(f'Each record must contain {sorted(REQUIRED_FIELDS)}.')
if len({item['chunk_id'] for item in raw_chunks}) != len(raw_chunks): raise ValueError('chunk_id values must be unique for FAISS indexing.')
if any(not isinstance(item['text'], str) or not item['text'].strip() for item in raw_chunks): raise ValueError('Every text value must be a non-empty string.')

@dataclass(frozen=True)
class JsonDocumentChunk:
    """Notebook-only attribute adapter for Salma's JSON records."""
    document_id: str
    chunk_id: str
    page_number: int
    section: str
    text: str
    chunk_index: int
    token_estimate: int

chunks = [JsonDocumentChunk(**item) for item in raw_chunks]
print(f'Validated {len(chunks)} chunks from {len({chunk.document_id for chunk in chunks})} document(s).')
print('Sections:', sorted({chunk.section for chunk in chunks}))

Validated 38 chunks from 1 document(s).
Sections: ['Abstract', 'Acknowledgements', 'Background', 'Conclusion', 'Front Matter', 'Introduction', 'References', 'Results']


In [4]:
import numpy as np
from backend.app.rag import FaissVectorStore, ScoreReranker, SemanticRetriever, SentenceTransformerEmbeddingProvider

MODEL_NAME, BATCH_SIZE = 'all-MiniLM-L6-v2', 32
provider = SentenceTransformerEmbeddingProvider(model_name=MODEL_NAME)
embedding_batches = [provider.embed_texts([chunk.text for chunk in chunks[start:start + BATCH_SIZE]]) for start in range(0, len(chunks), BATCH_SIZE)]
chunk_embeddings = np.vstack(embedding_batches).astype(np.float32)
store = FaissVectorStore(embedding_dimension=chunk_embeddings.shape[1])
store.add(chunks, chunk_embeddings)
faiss_retriever = SemanticRetriever(provider, store)
reranked_retriever = SemanticRetriever(provider, store, reranker=ScoreReranker())
print(f'Indexed {store.size} chunks using {MODEL_NAME}.')

c:\Users\Huawei\Downloads\Paper-PDF-Intelligent-Research-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5522.91it/s]


Indexed 38 chunks using all-MiniLM-L6-v2.


In [10]:
RESEARCH_QUESTIONS = ['Why do the authors use scaled dot-product attention?', 'How does multi-head attention work in the Transformer?', 'Which optimizer and learning-rate schedule did the authors use?', 'What machine-translation results did Transformer big achieve?']
TOP_K, CANDIDATE_K = 3, 20

def show_results(label, results):
    print(f'\n{label}')
    for rank, item in enumerate(results, 1):
        print(f'{rank}. {item.chunk_id} | {item.section} | p{item.page_number} | {item.score:.4f}')
        print('   ' + textwrap.shorten(item.text, width=260, placeholder=' …'))

for query in RESEARCH_QUESTIONS:
    print('\n' + '=' * 100 + f'\nQUERY: {query}')
    show_results('A. FAISS only', faiss_retriever.retrieve(query, top_k=TOP_K, candidate_k=CANDIDATE_K))
    show_results('B. FAISS + ScoreReranker', reranked_retriever.retrieve(query, top_k=TOP_K, candidate_k=CANDIDATE_K))


QUERY: Why do the authors use scaled dot-product attention?

A. FAISS only
1. chunk_e705395f | Background | p4 | 0.6569
   While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efﬁcient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms …
2. chunk_deda5b67 | Background | p3 | 0.6266
   This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i. 3.2 Attention An attention function can be described as mapping …
3. chunk_34f30dee | Background | p4 | 0.6071
   Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. query with all keys, divide each by √dk, and apply a softmax function to …

B. 

## Baseline metrics with manually judged gold chunks

These 16 cases are manually labeled against Salma's real chunk IDs. Recall is query-level: any returned gold chunk counts as a hit.

In [6]:
GOLD_CASES = {
    'What is the main Transformer proposal?': {'chunk_5a2523be'},
    'Why are recurrent models difficult to parallelize?': {'chunk_f3bbb2a0'},
    'What is self-attention?': {'chunk_59a941c4'},
    'How does the Transformer encoder-decoder architecture work?': {'chunk_f4e27088'},
    'What residual connections and normalization does the Transformer use?': {'chunk_1d2b56af'},
    'How is scaled dot-product attention computed?': {'chunk_34f30dee'},
    'Why are dot-product attention scores scaled by sqrt d_k?': {'chunk_e705395f'},
    'How does multi-head attention help the Transformer?': {'chunk_a43dced9', 'chunk_103c68f2'},
    'Where do queries keys and values come from in encoder self-attention?': {'chunk_75a34b39'},
    'Why do the authors use sinusoidal positional encodings?': {'chunk_50b65dae'},
    'Why is self-attention faster than recurrent layers?': {'chunk_497e6532', 'chunk_4e50dc69'},
    'What training data and batch size did the authors use?': {'chunk_7559dec7'},
    'Which optimizer and learning-rate schedule did the authors use?': {'chunk_e3503809'},
    'What machine-translation results did Transformer big achieve?': {'chunk_e7f9eeb8'},
    'What beam-search settings were used for inference?': {'chunk_ac2fcc3c'},
    'What is the conclusion of Attention Is All You Need?': {'chunk_318b1c4e'},
}
assert all(gold_id in {chunk.chunk_id for chunk in chunks} for ids in GOLD_CASES.values() for gold_id in ids)

def evaluate(name: str, retrieve: Callable[[str, int, int], list]) -> dict[str, float]:
    rankings = {query: [item.chunk_id for item in retrieve(query, top_k=10, candidate_k=20)] for query in GOLD_CASES}
    metrics = {f'Recall@{k}': sum(bool(set(rankings[q][:k]) & gold) for q, gold in GOLD_CASES.items()) / len(GOLD_CASES) for k in (3, 5, 10)}
    ranks = [next((rank for rank, item_id in enumerate(rankings[q], 1) if item_id in gold), None) for q, gold in GOLD_CASES.items()]
    metrics['MRR'] = sum(0.0 if rank is None else 1.0 / rank for rank in ranks) / len(ranks)
    metrics['Precision@3'] = sum(len(set(rankings[q][:3]) & gold) / 3 for q, gold in GOLD_CASES.items()) / len(GOLD_CASES)
    print(f'\n{name}')
    for metric, value in metrics.items(): print(f'{metric}: {value:.3f}')
    return metrics

faiss_metrics = evaluate('FAISS only', faiss_retriever.retrieve)
reranked_metrics = evaluate('FAISS + ScoreReranker', reranked_retriever.retrieve)
print('\nMetric delta (reranked - FAISS):')
for metric in faiss_metrics: print(f'{metric}: {reranked_metrics[metric] - faiss_metrics[metric]:+.3f}')


FAISS only
Recall@3: 0.812
Recall@5: 0.938
Recall@10: 1.000
MRR: 0.723
Precision@3: 0.292

FAISS + ScoreReranker
Recall@3: 0.812
Recall@5: 0.875
Recall@10: 1.000
MRR: 0.720
Precision@3: 0.292

Metric delta (reranked - FAISS):
Recall@3: +0.000
Recall@5: -0.062
Recall@10: +0.000
MRR: -0.004
Precision@3: +0.000


In [7]:
TRAINING_OPTIMIZER_QUERY = 'Which optimizer and learning-rate schedule did the authors use?'
TRAINING_OPTIMIZER_GOLD_ID = 'chunk_e3503809'
top_20 = faiss_retriever.retrieve(TRAINING_OPTIMIZER_QUERY, top_k=20, candidate_k=20)
top_20_ids = [item.chunk_id for item in top_20]
rank = top_20_ids.index(TRAINING_OPTIMIZER_GOLD_ID) + 1 if TRAINING_OPTIMIZER_GOLD_ID in top_20_ids else None
print(f'Gold optimizer chunk: {TRAINING_OPTIMIZER_GOLD_ID}')
print('FAISS top-20 rank:', rank if rank is not None else 'not retrieved')
for position, item in enumerate(top_20, 1):
    marker = ' <-- gold optimizer chunk' if item.chunk_id == TRAINING_OPTIMIZER_GOLD_ID else ''
    print(f'{position:>2}. {item.chunk_id} | {item.section} | p{item.page_number} | {item.score:.4f}{marker}')

Gold optimizer chunk: chunk_e3503809
FAISS top-20 rank: 1
 1. chunk_e3503809 | Background | p7 | 0.4948 <-- gold optimizer chunk
 2. chunk_c48a574f | References | p10 | 0.3405
 3. chunk_7559dec7 | Background | p7 | 0.3315
 4. chunk_ac2fcc3c | Results | p8 | 0.3265
 5. chunk_a310a18f | Background | p8 | 0.3148
 6. chunk_984c1047 | Results | p8 | 0.3145
 7. chunk_e7f9eeb8 | Results | p8 | 0.3105
 8. chunk_b9e386d8 | Background | p2 | 0.2991
 9. chunk_4e50dc69 | Background | p6 | 0.2956
10. chunk_e705395f | Background | p4 | 0.2896
11. chunk_8ced25c3 | References | p10 | 0.2840
12. chunk_f3bbb2a0 | Introduction | p2 | 0.2796
13. chunk_103c68f2 | Background | p5 | 0.2748
14. chunk_34f30dee | Background | p4 | 0.2653
15. chunk_497e6532 | Background | p6 | 0.2574
16. chunk_50b65dae | Background | p6 | 0.2445
17. chunk_4c44626c | Abstract | p1 | 0.2440
18. chunk_b3165422 | Introduction | p1 | 0.2387
19. chunk_5a2523be | Introduction | p2 | 0.2387
20. chunk_e44e177f | References | p11 | 0.2365

In [8]:
from backend.app.rag.bm25_store import Bm25Index

In [9]:
# Optional hybrid experiment: BM25 lexical retrieval + FAISS semantic retrieval + RRF.
# RRF uses sum(1 / (RRF_K + rank)) with one-based ranks.

RRF_K = 60
bm25_index = Bm25Index()
bm25_index.add(chunks)
hybrid_retriever = SemanticRetriever(provider, store, bm25_index=bm25_index, rrf_k=RRF_K)

def bm25_retrieve(query: str, top_k: int = 5, candidate_k: int = 20):
    return bm25_index.search(query, k=top_k)

def hybrid_retrieve(query: str, top_k: int = 5, candidate_k: int = 20):
    return hybrid_retriever.retrieve(query, top_k=top_k, candidate_k=candidate_k, strategy='hybrid_rrf')

METHODS = {
    'FAISS only': faiss_retriever.retrieve,
    'FAISS + ScoreReranker': reranked_retriever.retrieve,
    'BM25 only': bm25_retrieve,
    f'BM25 + FAISS + RRF (k={RRF_K})': hybrid_retrieve,
}

def evaluate_method(retrieve):
    rankings = {query: [item.chunk_id for item in retrieve(query, top_k=10, candidate_k=20)] for query in GOLD_CASES}
    metrics = {f'Recall@{k}': sum(bool(set(rankings[q][:k]) & gold) for q, gold in GOLD_CASES.items()) / len(GOLD_CASES) for k in (3, 5, 10)}
    ranks = [next((rank for rank, item_id in enumerate(rankings[q], 1) if item_id in gold), None) for q, gold in GOLD_CASES.items()]
    metrics['MRR'] = sum(0.0 if rank is None else 1.0 / rank for rank in ranks) / len(ranks)
    metrics['Precision@3'] = sum(len(set(rankings[q][:3]) & gold) / 3 for q, gold in GOLD_CASES.items()) / len(GOLD_CASES)
    return metrics

all_metrics = {name: evaluate_method(retrieve) for name, retrieve in METHODS.items()}
for name, metrics in all_metrics.items():
    print(f'\n{name}')
    for metric, value in metrics.items(): print(f'{metric}: {value:.3f}')

print('\nMetric deltas vs FAISS only:')
faiss_baseline = all_metrics['FAISS only']
for name, metrics in all_metrics.items():
    print(f'\n{name}')
    for metric, value in metrics.items(): print(f'{metric}: {value - faiss_baseline[metric]:+.3f}')

OPTIMIZER_QUERY = 'Which optimizer and learning-rate schedule did the authors use?'
OPTIMIZER_GOLD_ID = 'chunk_e3503809'
def rank_of(retrieve):
    ids = [item.chunk_id for item in retrieve(OPTIMIZER_QUERY, top_k=20, candidate_k=20)]
    return ids.index(OPTIMIZER_GOLD_ID) + 1 if OPTIMIZER_GOLD_ID in ids else None

print('\nOptimizer gold-chunk rank (top 20):')
for name in ('FAISS only', 'BM25 only', f'BM25 + FAISS + RRF (k={RRF_K})'):
    rank = rank_of(METHODS[name])
    rank_label = str(rank) if rank is not None else 'not retrieved'
    print(f'{name}: {rank_label}')


FAISS only
Recall@3: 0.812
Recall@5: 0.938
Recall@10: 1.000
MRR: 0.723
Precision@3: 0.292

FAISS + ScoreReranker
Recall@3: 0.812
Recall@5: 0.875
Recall@10: 1.000
MRR: 0.720
Precision@3: 0.292

BM25 only
Recall@3: 0.812
Recall@5: 0.875
Recall@10: 0.938
MRR: 0.733
Precision@3: 0.292

BM25 + FAISS + RRF (k=60)
Recall@3: 0.938
Recall@5: 0.938
Recall@10: 1.000
MRR: 0.809
Precision@3: 0.333

Metric deltas vs FAISS only:

FAISS only
Recall@3: +0.000
Recall@5: +0.000
Recall@10: +0.000
MRR: +0.000
Precision@3: +0.000

FAISS + ScoreReranker
Recall@3: +0.000
Recall@5: -0.062
Recall@10: +0.000
MRR: -0.004
Precision@3: +0.000

BM25 only
Recall@3: +0.000
Recall@5: -0.062
Recall@10: -0.062
MRR: +0.009
Precision@3: +0.000

BM25 + FAISS + RRF (k=60)
Recall@3: +0.125
Recall@5: +0.000
Recall@10: +0.000
MRR: +0.086
Precision@3: +0.042

Optimizer gold-chunk rank (top 20):
FAISS only: 1
BM25 only: 1
BM25 + FAISS + RRF (k=60): 1
